In [1]:
import sys
from pathlib import Path

current_dir = Path.cwd()

root_dir = current_dir
while not (root_dir / 'utils').is_dir() and root_dir != root_dir.parent:
    root_dir = root_dir.parent

if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

In [30]:
import numpy as np
import pandas as pd
from collections import Counter
import joblib
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from utils.custom_hyperparameter_tunning import CustomGridSearchCV
from utils.custom_cv import CustomKFold
from sklearn.utils import resample
from utils.model_manager import save_model_package

In [31]:
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, *, value=None):
        self.feature = feature       
        self.threshold = threshold   
        self.left = left             
        self.right = right           
        self.value = value           
        
    def is_leaf_node(self):
        return self.value is not None

In [33]:
class DecisionTreeScratch:
    def __init__(self, min_samples_split=2, max_depth=10, n_features=None):
        self.min_samples_split = min_samples_split
        self.max_depth = max_depth
        self.n_features = n_features
        self.root = None

    def fit(self, X, y):
        self.n_features = X.shape[1] if not self.n_features else min(X.shape[1], self.n_features)
        self.root = self._grow_tree(X, y)

    def _grow_tree(self, X, y, depth=0):
        n_samples, n_feats = X.shape
        n_labels = len(np.unique(y))

        if n_samples == 0:
            return Node(value=0)

        if (depth >= self.max_depth or n_labels == 1 or n_samples < self.min_samples_split):
            leaf_value = self._most_common_label(y)
            return Node(value=leaf_value)

        feat_idxs = np.random.choice(n_feats, self.n_features, replace=False)

        best_feature, best_thresh = self._best_split(X, y, feat_idxs)

        if best_feature is None:
            return Node(value=self._most_common_label(y))

        left_idxs, right_idxs = self._split(X[:, best_feature], best_thresh)
        left = self._grow_tree(X[left_idxs, :], y[left_idxs], depth + 1)
        right = self._grow_tree(X[right_idxs, :], y[right_idxs], depth + 1)
        
        return Node(best_feature, best_thresh, left, right)

    def _best_split(self, X, y, feat_idxs):
        best_gain = -1
        split_idx, split_threshold = None, None

        for feat_idx in feat_idxs:
            X_column = X[:, feat_idx]
            
            percentiles = np.percentile(X_column, [20, 40, 60, 80])
            thresholds = np.unique(percentiles) 

            for thr in thresholds:
                gain = self._information_gain(y, X_column, thr)
                if gain > best_gain:
                    best_gain = gain
                    split_idx = feat_idx
                    split_threshold = thr
                    
        return split_idx, split_threshold

    def _information_gain(self, y, X_column, threshold):
        parent_entropy = self._entropy(y)

        left_idxs, right_idxs = self._split(X_column, threshold)
        

        if len(left_idxs) == 0 or len(right_idxs) == 0:
            return -1 
        
        n = len(y)
        e_l, e_r = self._entropy(y[left_idxs]), self._entropy(y[right_idxs])
        child_entropy = (len(left_idxs) / n) * e_l + (len(right_idxs) / n) * e_r

        gain = parent_entropy - child_entropy
        return gain

    def _split(self, X_column, split_thresh):
        left_idxs = np.argwhere(X_column <= split_thresh).flatten()
        right_idxs = np.argwhere(X_column > split_thresh).flatten()
        return left_idxs, right_idxs

    def _entropy(self, y):
        hist = np.bincount(y)
        ps = hist / len(y)
        return -np.sum([p * np.log2(p) for p in ps if p > 0])

    def _most_common_label(self, y):
        counter = Counter(y)
        value = counter.most_common(1)[0][0]
        return value

    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X])

    def _traverse_tree(self, x, node):
        if node.is_leaf_node():
            return node.value
        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)

In [34]:
class RandomForestClassifierScratch:
    def __init__(self, n_estimators=10, max_depth=10, min_samples_split=2):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.trees = []

    def fit(self, X, y):
        if hasattr(X, "toarray"): 
            X = X.toarray()
        elif hasattr(X, "values"):
            X = X.values
        else:
            X = np.array(X)
            
        if hasattr(y, "values"):
            y = y.values
        else:
            y = np.array(y)
            
        self.trees = []
        for _ in range(self.n_estimators):
            X_samp, y_samp = self._bootstrap_samples(X, y)
            tree = DecisionTreeScratch(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                n_features=int(np.sqrt(X.shape[1]))
            )
            tree.fit(X_samp, y_samp)
            self.trees.append(tree)

    def _bootstrap_samples(self, X, y):
        n_samples = X.shape[0]
        idxs = np.random.choice(n_samples, n_samples, replace=True)
        return X[idxs], y[idxs]

    def predict(self, X):
        if hasattr(X, "toarray"): 
            X = X.toarray()
        elif hasattr(X, "values"):
            X = X.values
        else:
            X = np.array(X)
            
        tree_preds = np.array([tree.predict(X) for tree in self.trees])
        tree_preds = np.swapaxes(tree_preds, 0, 1)
        return np.array([Counter(pred).most_common(1)[0][0] for pred in tree_preds])

    def _most_common_label(self, y):
        counter = Counter(y)
        return counter.most_common(1)[0][0]

    def get_params(self, deep=True):
        return {
            "n_estimators": self.n_estimators,
            "max_depth": self.max_depth,
            "min_samples_split": self.min_samples_split
        }

    def set_params(self, **parameters):
        for parameter, value in parameters.items():
            setattr(self, parameter, value)
        return self

In [35]:
X_train = joblib.load('./data/ready_for_train/X_train_final.pkl')
X_test = joblib.load('./data/ready_for_train/X_test_final.pkl')
y_train = joblib.load('./data/ready_for_train/y_train.pkl')
y_test = joblib.load('./data/ready_for_train/y_test.pkl')

In [36]:
X_train_sub_sparse, y_train_sub_raw = resample(
    X_train, y_train, 
    n_samples=2000, 
    random_state=42, 
    stratify=y_train 
)

In [37]:
X_train_sub = X_train_sub_sparse.toarray()
y_train_sub = np.array(y_train_sub_raw)

In [38]:
cv = CustomKFold(n_splits=3, shuffle=True, random_state=42)

rf_model = RandomForestClassifierScratch(n_estimators=3)

param_grid = {
    'max_depth': [10, 20],
    'min_samples_split': [5, 10]
}

grid_search = CustomGridSearchCV(estimator=rf_model, param_grid=param_grid, cv=cv)
grid_search.fit(X_train_sub, y_train_sub)

Bắt đầu GridSearchCV: Sẽ chạy 4 tổ hợp tham số, mỗi tổ hợp 3 folds.
Tổng cộng số lần huấn luyện: 12 lần.

[1/4] Tham số: {'max_depth': 10, 'min_samples_split': 5} --> R2 trung bình: -0.0469
[2/4] Tham số: {'max_depth': 10, 'min_samples_split': 10} --> R2 trung bình: 0.0136
[3/4] Tham số: {'max_depth': 20, 'min_samples_split': 5} --> R2 trung bình: 0.3885
[4/4] Tham số: {'max_depth': 20, 'min_samples_split': 10} --> R2 trung bình: 0.3200
Tham số TỐT NHẤT: {'max_depth': 20, 'min_samples_split': 5}
R2 CV TỐT NHẤT  : 0.3885


In [39]:
best_params = grid_search.best_params_
final_rf = RandomForestClassifierScratch(
    n_estimators=15,
    max_depth=best_params['max_depth'],
    min_samples_split=best_params['min_samples_split']
)

In [40]:
final_rf.fit(X_train, y_train)

In [41]:
y_pred = final_rf.predict(X_test)

In [42]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

In [44]:
print(precision)
print(f1)

0.8061735261401557
0.8906298003072196


In [ ]:
metrics_rf_classification = {
    'Accuracy': accuracy,
    'Precision': precision,
    'Recall': recall,
    'F1_Score': f1
}

save_model_package(
    model=final_rf,                                  
    model_name="Random Forest Scratch",    
    best_params={'max_depth': best_params['max_depth'], 'min_samples_split': best_params['min_samples_split'], 'n_estimators': 15},
    metrics=metrics_rf_classification
)

'models\\Random_Forest_Scratch.pkl'